# supervised learning — Python demo

Numerical companion to the entry [supervised learning](https://dictionaryofml.org/terms/supervisedlearning.html) of the [Dictionary of Applied Machine Learning](https://dictionaryofml.org/): it recomputes what the entry states and prints one line per check.

Numerical companion to the glossary entry 'supervisedlearning'. The GeoSphere Austria weather station Krems (station id 3805) records the air temperature every hour; this script downloads the records for 2024 from the GeoSphere data hub (resource klima-v2-1h), keeps the reading at 06:00 (UTC) and the one at 15:00 (UTC) of each day, and writes the 365 days that carry both to supervisedlearning_weather.csv.

Requires NumPy and Matplotlib only, and uses fixed seeds, so the printed numbers reproduce exactly. Generated from [`pythondemos/supervisedlearning.py`](https://dictionaryofml.org/terms/supervisedlearning.py); CC BY 4.0.

Before running a downloaded copy, scan it: `uvx bandit -q -t B102,B307,B602,B605 supervisedlearning.py` (B102,B307,B602,B605 are the exec, eval and shell checks of [bandit](https://bandit.readthedocs.io/)) reports nothing: the script runs no shell command and evaluates no generated code.

In [ ]:
# Notebook shim: the script resolves output paths relative to __file__,
# which a notebook kernel does not define; everything lands in the
# working directory instead.
import os
__file__ = os.path.join(os.getcwd(), "supervisedlearning.py")
os.makedirs("pythondemos", exist_ok=True)

In [ ]:
"""Supervised learning on a year of days at Krems: the same days carry a
numeric label and a categorical one, so the same data points pose a
regression and a classification problem.

Purpose
-------
Numerical companion to the glossary entry 'supervisedlearning'.  The
GeoSphere Austria weather station Krems (station id 3805) records the air
temperature every hour; this script downloads the records for 2024 from
the GeoSphere data hub (resource klima-v2-1h), keeps the reading at 06:00
(UTC) and the one at 15:00 (UTC) of each day, and writes the 365 days
that carry both to supervisedlearning_weather.csv.

Ten of those days -- the 25th of each month from January to October --
are labeled twice, which is what the entry's figure shows side by side.
Ten days keep the figure readable; the day of the month was picked so
that both classes of the second labeling occur, and the dates are
stated in the caption so the selection is checkable:

  regression      the feature of a day is its 06:00 (UTC) reading and
                  the label is its 15:00 (UTC) reading, a number.  ERM
                  with the squared error loss over the polynomials of
                  degree two learns a hypothesis, drawn as a curve.

  classification  the features of a day are its two readings, and the
                  label is +1 if the next day is at least two degrees
                  warmer at 15:00 and -1 otherwise.

Both labels are known for a recorded day because both were measured,
which is what makes the setting supervised.  The two classes overlap
heavily, so the figure shows a labeled dataset and not a separable one.

Deterministic: the data are a fixed archive year and the hypothesis is
a least-squares solution.  Self-contained: numpy + matplotlib only
(stdlib urllib for the download).

Blocks
------
[B-data]    Download the hourly air temperature of 2024, keep the 06:00
            and the 15:00 (UTC) reading of each day, and keep the 25th of
            each month from January to October as ten labeled days.
[B-regress] The numeric label: ERM over the polynomials of degree two
            with the squared error loss.  Check that the learned
            hypothesis beats the best constant prediction.
[B-classify] The categorical label: +1 if the next day is at least two
            degrees warmer at 15:00, -1 otherwise.  Check that both classes
            occur and that no straight line separates them, searched
            exactly over directions and offsets rather than by fitting
            one line.
[B-fig]     Preview: the two labelings of the same days, side by side.

Outputs
-------
supervisedlearning_weather.csv : date, t06, t15 for the 365 days
supervisedlearning_days.csv    : t06, t15 of the ten labeled days
supervisedlearning_fit.csv     : t06, t15 of the learned hypothesis
supervisedlearning_pos.csv     : t06, t15 of the days labeled +1
supervisedlearning_neg.csv     : t06, t15 of the days labeled -1
supervisedlearning.png         : preview (checking only)
"""

# /// script
# requires-python = ">=3.10"
# dependencies = ["matplotlib==3.10.7", "numpy==2.3.4"]
# ///

import json
import urllib.request
from pathlib import Path

import numpy as np
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUT_DIR = Path(__file__).parent

report = []


def check(name, ok):
    report.append((name, bool(ok)))
    print(f"  [{'ok' if ok else 'FAIL'}] {name}")

**[B-data]** Download the hourly air temperature of 2024, keep the 06:00 and the 15:00 (UTC) reading of each day, and keep the 25th of each month from January to October as ten labeled days.

In [ ]:
URL = ("https://dataset.api.hub.geosphere.at/v1/station/historical/"
       "klima-v2-1h?parameters=tl&station_ids=3805"
       "&start=2024-01-01T00:00&end=2024-12-31T23:00")
with urllib.request.urlopen(URL, timeout=300) as resp:
    payload = json.load(resp)
readings = payload["features"][0]["properties"]["parameters"]["tl"]["data"]
byday = {}
for stamp, value in zip(payload["timestamps"], readings):
    if value is None:                        # the archive drops whole days
        continue
    hour = stamp[11:16]                      # the stamps are UTC, and so are
    if hour in ("06:00", "15:00"):           # the two times
        byday.setdefault(stamp[:10], {})[hour] = float(value)
record = [(d, r["06:00"], r["15:00"])
          for d, r in sorted(byday.items()) if len(r) == 2]
stamps = [d for d, _, _ in record]
t06 = np.array([a for _, a, _ in record])
t15 = np.array([b for _, _, b in record])
with open(OUT_DIR / "supervisedlearning_weather.csv", "w") as f:
    f.write("date,t06,t15\n")
    for day, lo, hi in zip(stamps, t06, t15):
        f.write(f"{day},{lo},{hi}\n")
check("[B-data] 365 of the days of 2024 carry both readings (the hourly "
      "record drops 2024-01-12)", len(t06) == 365)
check("[B-data] the record matches the archive (Jan 1: 4.0 at 06:00 and "
      "8.0 at 15:00, both UTC)",
      stamps[0] == "2024-01-01" and np.isclose(t06[0], 4.0)
      and np.isclose(t15[0], 8.0))

# the 25th of each month from January to October; each is followed in the
# record by the 26th, which the categorical label below needs
keep = [i for i, day in enumerate(stamps) if day[8:10] == "25"][:10]
check("[B-data] each labeled day is followed by the next calendar day",
      all(stamps[i + 1] == stamps[i][:8] + "26" for i in keep))
x, y = t06[keep], t15[keep]
nxt = t15[np.array(keep) + 1]
m = len(x)
np.savetxt(OUT_DIR / "supervisedlearning_days.csv", np.column_stack([x, y]),
           delimiter=",", header="t06,t15", comments="", fmt="%.1f")
check("[B-data] ten days carry both labels", m == 10)
check("[B-data] the ten days are the 25th of January to October",
      [stamps[i][:7] for i in keep]
      == [f"2024-{mth:02d}" for mth in range(1, 11)])

**[B-regress]** The numeric label: ERM over the polynomials of degree two with the squared error loss. Check that the learned hypothesis beats the best constant prediction.

In [ ]:
coeff = np.polyfit(x, y, 2)
curve_x = np.linspace(x.min(), x.max(), 120)
np.savetxt(OUT_DIR / "supervisedlearning_fit.csv",
           np.column_stack([curve_x, np.polyval(coeff, curve_x)]),
           delimiter=",", header="t06,t15", comments="", fmt="%.3f")
learned = float(((y - np.polyval(coeff, x)) ** 2).mean())
constant = float(((y - y.mean()) ** 2).mean())
check("[B-regress] the learned hypothesis beats the constant prediction",
      learned < constant)
print(f"  average squared error loss: {learned:.1f} for the learned "
      f"hypothesis against {constant:.1f} for the best constant")

**[B-classify]** The categorical label: +1 if the next day is at least two degrees warmer at 15:00, -1 otherwise. Check that both classes occur and that no straight line separates them, searched exactly over directions and offsets rather than by fitting one line.

In [ ]:
label = np.where(nxt - y >= 2.0, 1, -1)
X = np.column_stack([x, y])
np.savetxt(OUT_DIR / "supervisedlearning_pos.csv", X[label == 1],
           delimiter=",", header="t06,t15", comments="", fmt="%.1f")
np.savetxt(OUT_DIR / "supervisedlearning_neg.csv", X[label == -1],
           delimiter=",", header="t06,t15", comments="", fmt="%.1f")
check("[B-classify] the label takes only the two values", 
      set(np.unique(label)) == {-1, 1})
check("[B-classify] both classes occur among the ten days",
      min((label == 1).sum(), (label == -1).sum()) >= 3)
print(f"  {int((label == 1).sum())} of the ten days are followed by one at "
      f"least two degrees warmer, {int((label == -1).sum())} are not")

# no straight line separates them, searched exactly: every direction, and
# every offset that matters, is one of the finitely many that put a
# different subset of the days on each side
best = 0.0
for angle in np.linspace(0.0, np.pi, 2000, endpoint=False):
    scores = X @ np.array([np.cos(angle), np.sin(angle)])
    for offset in np.unique(scores):
        for side in (1.0, -1.0):
            hit = (np.sign(side * (scores - offset + 1e-12)) == label).mean()
            best = max(best, float(hit))
check("[B-classify] no straight line separates the two classes", best < 1.0)
print(f"  the best straight line gets {best:.0%} of the ten days right")

**[B-fig]** Preview: the two labelings of the same days, side by side.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.0, 4.2))
ax1.plot(x, y, "o", ms=5, mfc="none", color="black", label="day")
ax1.plot(curve_x, np.polyval(coeff, curve_x), "-", lw=1.6, color="black",
         label="learned hypothesis")
ax1.set_xlabel("feature: the 06:00 (UTC) reading (degrees C)")
ax1.set_ylabel("label: the 15:00 (UTC) reading (degrees C)")
ax1.set_title("(a) a numeric label: regression", y=-0.30)
ax1.legend(frameon=False, fontsize=8, loc="upper left")
ax2.plot(X[label == -1, 0], X[label == -1, 1], "s", ms=5, mfc="none",
         color="black", label="next day not 2 deg warmer at 15:00")
ax2.plot(X[label == 1, 0], X[label == 1, 1], "o", ms=5, color="black",
         label="next day at least 2 deg warmer at 15:00")
ax2.set_xlabel("feature: daily minimum temperature (degrees C)")
ax2.set_ylabel("feature: daily maximum temperature (degrees C)")
ax2.set_title("(b) a categorical label: classification", y=-0.30)
ax2.legend(frameon=False, fontsize=8, loc="upper left")
fig.tight_layout()
fig.savefig(OUT_DIR / "supervisedlearning.png", dpi=110)
plt.close(fig)
check("[B-fig] the preview was written",
      (OUT_DIR / "supervisedlearning.png").exists())

bad = [n for n, ok in report if not ok]
print(f"\n{len(report) - len(bad)}/{len(report)} checks pass")
if bad:
    raise SystemExit("FAILED: " + "; ".join(bad))